### Using ChromaDB for the vector database and wrapping the final response with an LLM is a classic, highly effective RAG (Retrieval-Augmented Generation) setup for customer support.

# Phase 1 : Data Understanding && Data Cleaning

In [8]:
import pandas as pd
import re

def clean_customer_support_data(file_path: str) -> pd.DataFrame:
    # 1. Load Dataset
    print("Loading dataset...")
    df = pd.read_csv(file_path)
    initial_rows = len(df)
    
    # 2. Remove duplicate tickets based on unique identifier
    df = df.drop_duplicates(subset=['Ticket ID'])
    print(f"Removed {initial_rows - len(df)} duplicate rows.")
    
    # 3. Handle missing values
    # Fill categorical/text missing values with logical placeholders
    text_cols = ['Ticket Subject', 'Ticket Description', 'Resolution', 'Product Purchased', 'Ticket Type']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown")
            
    # Fill numerical/categorical missing values with defaults if any exist
    if 'Customer Satisfaction Rating' in df.columns:
        df['Customer Satisfaction Rating'] = df['Customer Satisfaction Rating'].fillna(-1) # -1 indicates unrated
        
    # 4. Text Normalization Function
    def normalize_text(text):
        if not isinstance(text, str):
            return ""
        # Remove unnecessary special punctuation but keep basic sentence structure
        text = re.sub(r'[^\w\s\.\?\!,]', '', text)
        # Normalize whitespace (replace multiple spaces/newlines with a single space)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    
    print("Normalizing text fields...")
    df['Ticket Subject'] = df['Ticket Subject'].apply(normalize_text)
    df['Ticket Description'] = df['Ticket Description'].apply(normalize_text)
    df['Resolution'] = df['Resolution'].apply(normalize_text)
    
    # 5. Standardize categories (Product Names and Ticket Types)
    print("Standardizing category names...")
    df['Product Purchased'] = df['Product Purchased'].astype(str).str.strip().str.title()
    df['Ticket Type'] = df['Ticket Type'].astype(str).str.strip().str.title()
    df['Ticket Status'] = df['Ticket Status'].astype(str).str.strip().str.title()
    df['Ticket Priority'] = df['Ticket Priority'].astype(str).str.strip().str.title()
    df['Ticket Channel'] = df['Ticket Channel'].astype(str).str.strip().str.title()
    
    # 6. Convert dates into a single unified format (YYYY-MM-DD)
    print("Standardizing date format...")
    if 'Date of Purchase' in df.columns:
        df['Date of Purchase'] = pd.to_datetime(df['Date of Purchase'], errors='coerce')
        # Format as string YYYY-MM-DD for clean JSON storage in ChromaDB metadata
        df['Date of Purchase'] = df['Date of Purchase'].dt.strftime('%Y-%m-%d').fillna("Unknown")
        
    print(f"Data cleaning complete. Final row count: {len(df)}")
    return df


# Create Documents

In [9]:
import pandas as pd
from typing import Tuple, List, Dict, Any

def prepare_chromadb_payload(df: pd.DataFrame) -> Tuple[List[str], List[Dict[str, Any]], List[str]]:
    """
    Transforms a cleaned DataFrame into ChromaDB-compatible components.
    Returns: (documents, metadatas, ids)
    """
    documents = []
    metadatas = []
    ids = []
    
    print(f"Creating semantic text documents for {len(df)} records...")
    
    for _, row in df.iterrows():
        # 1. Build the unified searchable text content
        doc_text = (
            f"Ticket Subject: {row['Ticket Subject']}\n"
            f"Product: {row['Product Purchased']}\n"
            f"Issue Description: {row['Ticket Description']}\n"
            f"Resolution: {row['Resolution']}"
        )
        documents.append(doc_text)
        
        # 2. Construct the metadata dict (ChromaDB supports str, int, float, or bool)
        metadata = {
            "ticket_id": str(row['Ticket ID']),
            "customer_age": int(row['Customer Age']) if pd.notna(row['Customer Age']) else -1,
            "customer_gender": str(row['Customer Gender']),
            "product_purchased": str(row['Product Purchased']),
            "ticket_type": str(row['Ticket Type']),
            "ticket_status": str(row['Ticket Status']),
            "ticket_priority": str(row['Ticket Priority']),
            "ticket_channel": str(row['Ticket Channel']),
            "customer_satisfaction_rating": int(row['Customer Satisfaction Rating']),
            "date_of_purchase": str(row['Date of Purchase'])
        }
        metadatas.append(metadata)
        
        # 3. Use Ticket ID as the explicit unique identifier
        ids.append(str(row['Ticket ID']))
        
    return documents, metadatas, ids


# Store Metadata Separately

In [10]:
import chromadb
from chromadb.utils import embedding_functions
from typing import List, Dict, Any

def store_in_chromadb(
    documents: List[str], 
    metadatas: List[Dict[str, Any]], 
    ids: List[str], 
    db_path: str = "./chroma_db", 
    collection_name: str = "customer_tickets"
):
    """
    Initializes a local persistent ChromaDB client, creates a collection,
    and batches the insertion of documents and separate metadata filters.
    """
    # 1. Initialize Persistent Client (saves data to a local directory)
    print(f"Initializing persistent ChromaDB client at: {db_path}")
    client = chromadb.PersistentClient(path=db_path)
    
    # 2. Define the Embedding Function
    # By default, Chroma uses All-MiniLM-L6-v2, which runs locally and is lightweight.
    default_ef = embedding_functions.DefaultEmbeddingFunction()
    
    # 3. Create or Get the Collection
    print(f"Setting up collection: '{collection_name}'...")
    collection = client.get_or_create_collection(
        name=collection_name, 
        embedding_function=default_ef
    )
    
    # 4. Batch Upsert (ChromaDB has an upper limit of ~41,000 items per batch,
    # but small batches of 500-1000 keep memory footprint low).
    batch_size = 500
    total_docs = len(documents)
    print(f"Upserting {total_docs} records in batches of {batch_size}...")
    
    for i in range(0, total_docs, batch_size):
        end_idx = min(i + batch_size, total_docs)
        
        batch_ids = ids[i:end_idx]
        batch_docs = documents[i:end_idx]
        batch_meta = metadatas[i:end_idx]
        
        # Using upsert instead of add ensures you can safely re-run this script
        # without throwing duplicate ID errors.
        collection.upsert(
            ids=batch_ids,
            documents=batch_docs,
            metadatas=batch_meta
        )
        print(f"Progress: {end_idx}/{total_docs} inserted.")
        
    print(f"Successfully populated collection '{collection_name}'!")
    return collection


#  Embedding Generation

In [11]:
import chromadb
from chromadb.utils import embedding_functions

# 1. Initialize the embedding function explicitly
# This downloads and initializes 'all-MiniLM-L6-v2' automatically on its first run
embedding_model = embedding_functions.DefaultEmbeddingFunction()

# --- OPTIONAL: Debugging/Testing Vectors ---
def inspect_dense_vectors(sample_text: str):
    """
    Convenience function to see what a dense vector looks like.
    """
    # Convert text to a dense vector manually
    vector_output = embedding_model([sample_text])
    
    dense_vector = vector_output[0]
    print(f"Text Input: '{sample_text}'")
    print(f"Vector Dimensions (Length): {len(dense_vector)}")
    print(f"First 5 vector values: {dense_vector[:5]}")

# Try inspecting a sample ticket format
inspect_dense_vectors("Ticket Subject: Screen flickering\nProduct: Laptop\nIssue Description: Screen blink")
# --------------------------------------------

# 2. Bind the model directly to your Collection
def create_vector_collection(client: chromadb.PersistentClient, collection_name: str):
    """
    Creates a collection bound to the chosen embedding model.
    Any text passed here will be automatically vectorized.
    """
    collection = client.get_or_create_collection(
        name=collection_name,
        embedding_function=embedding_model,
        metadata={"hnsw:space": "cosine"} # Sets distance metric to Cosine Similarity
    )
    return collection

Text Input: 'Ticket Subject: Screen flickering
Product: Laptop
Issue Description: Screen blink'
Vector Dimensions (Length): 384
First 5 vector values: [-0.06462827  0.03192259  0.05803147 -0.04320494 -0.00064762]


# Store in ChromaDB

In [12]:
import os
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

def run_full_ingestion_pipeline(csv_path: str, db_path: str = "./chroma_db", collection_name: str = "customer_tickets"):
    """
    Executes the complete pipeline: Loads, cleans, builds structured documents,
    extracts metadata filters, generates dense vectors, and stores everything in ChromaDB.
    """
    # --- STEP 1 & 2: Load and Clean Data ---
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Could not find the dataset at {csv_path}")
        
    print("Step 1 & 2: Loading and cleaning customer support dataset...")
    df = pd.read_csv(csv_path)
    df = df.drop_duplicates(subset=['Ticket ID'])
    
    # Fill missing values
    text_cols = ['Ticket Subject', 'Ticket Description', 'Resolution', 'Product Purchased', 'Ticket Type']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown")
    if 'Customer Satisfaction Rating' in df.columns:
        df['Customer Satisfaction Rating'] = df['Customer Satisfaction Rating'].fillna(-1)
        
    # Standardize string categories
    for col in ['Product Purchased', 'Ticket Type', 'Ticket Status', 'Ticket Priority', 'Ticket Channel']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.title()
            
    # Standardize date format
    if 'Date of Purchase' in df.columns:
        df['Date of Purchase'] = pd.to_datetime(df['Date of Purchase'], errors='coerce')
        df['Date of Purchase'] = df['Date of Purchase'].dt.strftime('%Y-%m-%d').fillna("Unknown")

    # --- STEP 3 & 4: Separate into Documents and Metadata Filters ---
    print("\nStep 3 & 4: Separating searchable documents from metadata filters...")
    documents = []
    metadatas = []
    ids = []
    
    for _, row in df.iterrows():
        # The searchable text chunk
        doc_text = (
            f"Ticket Subject: {row['Ticket Subject']}\n"
            f"Product: {row['Product Purchased']}\n"
            f"Issue Description: {row['Ticket Description']}\n"
            f"Resolution: {row['Resolution']}"
        )
        documents.append(doc_text)
        
        # The isolated query filters
        metadata = {
            "ticket_id": str(row['Ticket ID']),
            "customer_age": int(row['Customer Age']) if pd.notna(row['Customer Age']) else -1,
            "customer_gender": str(row['Customer Gender']),
            "product_purchased": str(row['Product Purchased']),
            "ticket_type": str(row['Ticket Type']),
            "ticket_status": str(row['Ticket Status']),
            "ticket_priority": str(row['Ticket Priority']),
            "ticket_channel": str(row['Ticket Channel']),
            "customer_satisfaction_rating": int(row['Customer Satisfaction Rating']),
            "date_of_purchase": str(row['Date of Purchase'])
        }
        metadatas.append(metadata)
        ids.append(str(row['Ticket ID']))

    # --- STEP 5 & 6: Initialize DB, Generate Embeddings, and Store ---
    print(f"\nStep 5 & 6: Initializing ChromaDB and generating dense vectors...")
    client = chromadb.PersistentClient(path=db_path)
    
    # Use the lightweight, local all-MiniLM-L6-v2 embedding model
    embedding_model = embedding_functions.DefaultEmbeddingFunction()
    
    collection = client.get_or_create_collection(
        name=collection_name, 
        embedding_function=embedding_model,
        metadata={"hnsw:space": "cosine"} # Use cosine distance for semantic relevance
    )
    
    # Store items in safe batches
    batch_size = 500
    total_docs = len(documents)
    
    for i in range(0, total_docs, batch_size):
        end_idx = min(i + batch_size, total_docs)
        
        # collection.upsert automatically links ID, calculates the Vector Embedding 
        # from the Document, and appends the Metadata block.
        collection.upsert(
            ids=ids[i:end_idx],
            documents=documents[i:end_idx],
            metadatas=metadatas[i:end_idx]
        )
        print(f"Stored records {end_idx}/{total_docs} successfully.")
        
    print(f"\nPipeline execution complete! Collection '{collection_name}' is fully loaded.")
    return collection

# CLI Chatbot && Query Embedding

In [13]:
import chromadb
from chromadb.utils import embedding_functions
from typing import List, Dict, Any, Optional

class TicketSearchEngine:
    def __init__(self, db_path: str = "./chroma_db", collection_name: str = "customer_tickets"):
        """
        Connects to the persistent database and binds the embedding model
        to translate user questions into dense vectors automatically.
        """
        self.client = chromadb.PersistentClient(path=db_path)
        self.embedding_model = embedding_functions.DefaultEmbeddingFunction()
        self.collection = self.client.get_collection(
            name=collection_name, 
            embedding_function=self.embedding_model
        )

    def search(self, user_query: str, n_results: int = 3, metadata_filter: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
        """
        Converts user_query to an embedding and retrieves the most semantically 
        similar tickets from ChromaDB, applying filters if provided.
        """
        # Under the hood, ChromaDB passes user_query through self.embedding_model
        # to generate the query embedding, then runs a Cosine Similarity match.
        results = self.collection.query(
            query_texts=[user_query],
            n_results=n_results,
            where=metadata_filter
        )
        
        # Format the flat arrays into a friendly, structured list of dictionaries
        formatted_results = []
        if results and results['documents']:
            for i in range(len(results['documents'][0])):
                formatted_results.append({
                    "id": results['ids'][0][i],
                    "document": results['documents'][0][i],
                    "metadata": results['metadatas'][0][i],
                    "distance": results['distances'][0][i] # Cosine distance
                })
        return formatted_results


def run_cli_chatbot():
    """
    Launches the Phase 7 interactive CLI loop.
    """
    print("=" * 60)
    print("🤖 CUSTOMER SUPPORT KNOWLEDGE BASE - SEMANTIC SEARCH CLI")
    print("=" * 60)
    print("Connecting to vector database...")
    
    try:
        engine = TicketSearchEngine()
        print("Connected! Type your question below. (Type 'exit' or 'quit' to stop)\n")
    except Exception as e:
        print(f"Error connecting to database: {e}")
        print("Please ensure you ran the Phase 6 ingestion pipeline first.")
        return

    while True:
        try:
            user_input = input("\nEnter your query: ").strip()
            if not user_input:
                continue
            if user_input.lower() in ['exit', 'quit']:
                print("Goodbye!")
                break
                
            print(f"🔄 Vectorizing query and retrieving top matches...")
            matches = engine.search(user_query=user_input, n_results=2)
            
            if not matches:
                print("No relevant tickets found.")
                continue
                
            print(f"\n🔍 Found {len(matches)} highly relevant past tickets:\n")
            for idx, match in enumerate(matches, 1):
                print(f"--- [Match #{idx}] Ticket ID: {match['id']} (Distance: {match['distance']:.4f}) ---")
                print(f"Status: {match['metadata']['ticket_status']} | Priority: {match['metadata']['ticket_priority']} | Product: {match['metadata']['product_purchased']}")
                print(f"{match['document']}")
                print("-" * 60)
                
        except KeyboardInterrupt:
            print("\nExiting...")
            break

if __name__ == "__main__":
    # To run the CLI search engine directly:
    run_cli_chatbot()

🤖 CUSTOMER SUPPORT KNOWLEDGE BASE - SEMANTIC SEARCH CLI
Connecting to vector database...
Error connecting to database: Collection [customer_tickets] does not exist
Please ensure you ran the Phase 6 ingestion pipeline first.


# Semantic Search

In [14]:
import chromadb
from chromadb.utils import embedding_functions
from typing import List, Dict, Any, Optional

class CustomerSupportSemanticSearcher:
    def __init__(self, db_path: str = "./chroma_db", collection_name: str = "customer_tickets"):
        """
        Connects to your local ChromaDB persistent instance.
        """
        self.client = chromadb.PersistentClient(path=db_path)
        self.embedding_function = embedding_functions.DefaultEmbeddingFunction()
        
        # Connect to the collection populated during Phase 6
        self.collection = self.client.get_collection(
            name=collection_name,
            embedding_function=self.embedding_function
        )

    def retrieve_similar_tickets(
        self, 
        user_query: str, 
        top_k: int = 4, 
        status_filter: Optional[str] = None,
        product_filter: Optional[str] = None
    ) -> List[Dict[str, Any]]:
        """
        Executes Phase 9 semantic retrieval. Converts the user query into dense vectors,
        searches ChromaDB, and yields the top K matching reference tickets.
        """
        # 1. Build optional runtime metadata filters (Pre-filtering)
        where_clause = {}
        filters = []
        
        if status_filter:
            filters.append({"ticket_status": status_filter.strip().title()})
        if product_filter:
            filters.append({"product_purchased": product_filter.strip().title()})
            
        if len(filters) == 1:
            where_clause = filters[0]
        elif len(filters) > 1:
            where_clause = {"$and": filters}
        else:
            where_clause = None

        # 2. Query ChromaDB (Triggers implicit Phase 8 Query Vectorization)
        results = self.collection.query(
            query_texts=[user_query],
            n_results=top_k,
            where=where_clause
        )
        
        # 3. Structure the returned data for your upcoming LLM pipeline
        formatted_context_records = []
        
        if results and results['documents'] and len(results['documents'][0]) > 0:
            for i in range(len(results['documents'][0])):
                record = {
                    "id": results['ids'][0][i],
                    "document": results['documents'][0][i],
                    "metadata": results['metadatas'][0][i],
                    "cosine_distance": results['distances'][0][i]
                }
                formatted_context_records.append(record)
                
        return formatted_context_records

# --- Quick Test Verification ---
if __name__ == "__main__":
    searcher = CustomerSupportSemanticSearcher()
    
    # Example: Look for past solutions for screen glitches specifically on Closed tickets
    test_query = "The monitor display keeps flickering black every couple of seconds"
    print(f"Testing Semantic Search for: '{test_query}'...\n")
    
    hits = searcher.retrieve_similar_tickets(
        user_query=test_query, 
        top_k=3, 
        status_filter="Closed"
    )
    
    for rank, hit in enumerate(hits, 1):
        print(f"Rank {rank} | Ticket ID: {hit['id']} (Distance: {hit['cosine_distance']:.4f})")
        print(f"Metadata: {hit['metadata']['product_purchased']} | Rating: {hit['metadata']['customer_satisfaction_rating']}/5")
        print(f"Content:\n{hit['document']}")
        print("="*50)

NotFoundError: Collection [customer_tickets] does not exist

# Context Construction

In [15]:
import os
from openai import OpenAI
from typing import List, Dict, Any

class RagResponseGenerator:
    def __init__(self, model_name: str = "gpt-4o-mini"):
        """
        Initializes the LLM generation client.
        Expects your OPENAI_API_KEY to be set in your system environment.
        """
        # The client automatically searches for os.environ.get("OPENAI_API_KEY")
        self.client = OpenAI()
        self.model_name = model_name

    def construct_context_prompt(self, user_query: str, retrieved_tickets: List[Dict[str, Any]]) -> List[Dict[str, str]]:
        """
        Executes Phase 10: Compiles retrieved tickets safely into a clean prompt text
        and constructs the payload structure for the LLM.
        """
        # 1. Build the contextual grounding text block from the historical records
        context_blocks = []
        for idx, ticket in enumerate(retrieved_tickets, 1):
            block = (
                f"--- HISTORICAL REFERENCE TICKET #{idx} ---\n"
                f"Ticket ID: {ticket['id']}\n"
                f"Product Category: {ticket['metadata'].get('product_purchased', 'Unknown')}\n"
                f"Historical Ticket Content:\n{ticket['document']}\n"
                f"Customer Satisfaction Rating Given: {ticket['metadata'].get('customer_satisfaction_rating', 'N/A')}/5\n"
            )
            context_blocks.append(block)
            
        combined_context = "\n".join(context_blocks)

        # 2. Frame systemic boundary rules for the customer assistant
        system_instructions = (
            "You are an expert internal customer support AI assistant.\n"
            "Your objective is to help agents answer user queries accurately by extracting facts "
            "solely from the historical reference tickets provided below.\n\n"
            "CRITICAL RULES:\n"
            "1. Ground your reasoning ONLY on the facts present in the provided context tickets.\n"
            "2. If the context tickets do not contain relevant solutions to resolve the user's problem, "
            "explicitly state that you could not find a confirmed resolution in past data, and offer a basic "
            "troubleshooting step for that specific product category instead.\n"
            "3. Maintain a professional, empathetic, and concise technical support tone."
        )

        user_content = (
            f"Here are the most semantically relevant historical tickets found in our database:\n\n"
            f"{combined_context}\n"
            f"--- END OF CONTEXT ---\n\n"
            f"Current User Query: {user_query}\n\n"
            f"Please provide a structured response or solution based strictly on the relevant historical data above:"
        )

        # Return structured chat completion schema arrays
        return [
            {"role": "system", "content": system_instructions},
            {"role": "user", "content": user_content}
        ]

    def generate_answer(self, user_query: str, retrieved_tickets: List[Dict[str, Any]]) -> str:
        """
        Pipes the structured prompt payload directly through the LLM.
        """
        if not retrieved_tickets:
            return "I couldn't find any historical records matching your request to ground my response."

        # Compile the context payload
        messages_payload = self.construct_context_prompt(user_query, retrieved_tickets)
        
        try:
            # Call OpenAI Chat Completions endpoint
            completion = self.client.chat.completions.create(
                model=self.model_name,
                messages=messages_payload,
                temperature=0.3 # Low temperature guarantees more precise factual matching
            )
            return completion.choices[0].message.content
        except Exception as e:
            return f"An error occurred while generating the response via LLM: {str(e)}"

# Prompt Engineering

In [ ]:
import os
import re
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
from typing import List, Dict, Any, Optional

# =====================================================================
# PHASES 1-6: STORAGE & INGESTION HOOKS (For reference/initialization)
# =====================================================================
def run_ingestion_if_needed(csv_path: str, db_path: str, collection_name: str):
    """Checks if the Chroma collection exists; if not, builds it from CSV."""
    client = chromadb.PersistentClient(path=db_path)
    ef = embedding_functions.DefaultEmbeddingFunction()
    
    try:
        # Check if collection exists and has documents
        coll = client.get_collection(name=collection_name, embedding_function=ef)
        if coll.count() > 0:
            print(f"✓ Found existing collection '{collection_name}' with {coll.count()} records.")
            return coll
    except Exception:
        pass
        
    print("! Vector collection not found or empty. Beginning Phase 1-6 Data Ingestion...")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Could not locate dataset at {csv_path}. Please place it in the directory.")
        
    df = pd.read_csv(csv_path).drop_duplicates(subset=['Ticket ID'])
    
    # Quick fill and standardization matching your specs
    for col in ['Ticket Subject', 'Ticket Description', 'Resolution', 'Product Purchased', 'Ticket Type']:
        df[col] = df[col].fillna("Unknown")
    df['Customer Satisfaction Rating'] = df['Customer Satisfaction Rating'].fillna(-1)
    for col in ['Product Purchased', 'Ticket Type', 'Ticket Status', 'Ticket Priority']:
        df[col] = df[col].astype(str).str.strip().str.title()
    
    documents, metadatas, ids = [], [], []
    for _, row in df.iterrows():
        documents.append(
            f"Ticket Subject: {row['Ticket Subject']}\n"
            f"Product: {row['Product Purchased']}\n"
            f"Issue Description: {row['Ticket Description']}\n"
            f"Resolution: {row['Resolution']}"
        )
        metadatas.append({
            "ticket_status": str(row['Ticket Status']),
            "product_purchased": str(row['Product Purchased']),
            "ticket_priority": str(row['Ticket Priority']),
            "customer_satisfaction_rating": int(row['Customer Satisfaction Rating'])
        })
        ids.append(str(row['Ticket ID']))
        
    coll = client.get_or_create_collection(name=collection_name, embedding_function=ef, metadata={"hnsw:space": "cosine"})
    
    batch_size = 500
    for i in range(0, len(documents), batch_size):
        end = min(i + batch_size, len(documents))
        coll.upsert(ids=ids[i:end], documents=documents[i:end], metadatas=metadatas[i:end])
    print(f"✓ Successfully ingested {len(documents)} tickets into ChromaDB.")
    return coll


# =====================================================================
# PHASES 7-11: RETRIEVAL & PROMPT ENGINEERING ENGINE
# =====================================================================
class RagSupportEngine:
    def __init__(self, db_path: str = "./chroma_db", collection_name: str = "customer_tickets"):
        self.client = chromadb.PersistentClient(path=db_path)
        self.ef = embedding_functions.DefaultEmbeddingFunction()
        self.collection = self.client.get_collection(name=collection_name, embedding_function=self.ef)
        self.llm_client = OpenAI() # Expects OPENAI_API_KEY environment variable

    def get_system_prompt(self) -> str:
        """
        Phase 11: The Engineered System Prompt.
        Defines precise agent persona, operational constraints, and fallback behavior.
        """
        return (
            "You are an elite corporate customer support tier-2 technician.\n"
            "Your purpose is to resolve the user's issue by analyzing the provided historical ticket references.\n\n"
            "CRITICAL BEHAVIORAL GUARDRAILS:\n"
            "1. GROUNDING RULE: You must rely ONLY on facts explicitly stated in the context documents below. Do not invent details.\n"
            "2. SATISFACTION FILTER: Prioritize resolutions that came from reference tickets with high Customer Satisfaction Ratings (4 or 5).\n"
            "3. FALLBACK PROTOCOL: If the provided reference tickets contain no relevant resolution to fix the user's problem, do not hallucinate a fix. "
            "Instead, reply with: 'I am sorry, but our historical records do not contain a verified solution for this specific problem.' "
            "Then offer exactly 2 standard generic troubleshooting steps appropriate for that product category.\n"
            "4. STYLE: Be highly objective, technical, brief, and structured. Do not use generic pleasantries like 'I hope this helps!'."
        )

    def run_rag_pipeline(self, user_query: str) -> str:
        # Phase 8 & 9: Query Embedding & Semantic Search (Top 3 matches)
        results = self.collection.query(query_texts=[user_query], n_results=3)
        
        if not results or not results['documents'] or len(results['documents'][0]) == 0:
            return "No historical context found in database to synthesize a response."
            
        # Phase 10: Context Construction
        context_str = ""
        for i in range(len(results['documents'][0])):
            meta = results['metadatas'][0][i]
            context_str += (
                f"\n--- HISTORICAL RECORD #{i+1} ---\n"
                f"Product Category: {meta.get('product_purchased')}\n"
                f"Ticket Details:\n{results['documents'][0][i]}\n"
                f"Historical Customer Satisfaction Rating: {meta.get('customer_satisfaction_rating')}/5\n"
                f"---------------------------\n"
            )
            
        # Compile messages for the ChatCompletion API
        messages = [
            {"role": "system", "content": self.get_system_prompt()},
            {"role": "user", "content": f"HISTORICAL CONTEXT DOCUMENTS:\n{context_str}\n\nCURRENT USER ISSUE:\n{user_query}\n\nProvide your technical resolution:"}
        ]
        
        # Pass payload to the LLM
        try:
            response = self.llm_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                temperature=0.2 # Kept low to enforce strict adherence to context
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"LLM Generation Error: {str(e)}"


# =====================================================================
# PHASE 7: INTERACTIVE TERMINAL LOOP
# =====================================================================
def launch_chatbot():
    print("=" * 60)
    print("🤖 RAG SEMANTIC SEARCH CUSTOMER SUPPORT BOT READY")
    print("=" * 60)
    
    # Initialize / verify database states
    try:
        run_ingestion_if_needed(
            csv_path="customer_support_tickets.csv", 
            db_path="./chroma_db", 
            collection_name="customer_tickets"
        )
        engine = RagSupportEngine()
        print("\nSystem running completely locally. Enter your queries below.")
        print("Type 'exit' to close the terminal.\n")
    except Exception as e:
        print(f"System initialization failed: {e}")
        return

    while True:
        try:
            query = input("User 👤: ").strip()
            if not query:
                continue
            if query.lower() in ['exit', 'quit']:
                print("Ending support session. Goodbye!")
                break
                
            print("System ⚙️: Vectorizing input, running semantic matching, and asking LLM...")
            reply = engine.run_rag_pipeline(user_query=query)
            print(f"\nBot 🤖:\n{reply}\n")
            print("-" * 60)
            
        except KeyboardInterrupt:
            print("\nSession aborted.")
            break

if __name__ == "__main__":
    launch_chatbot()

🤖 RAG SEMANTIC SEARCH CUSTOMER SUPPORT BOT READY
! Vector collection not found or empty. Beginning Phase 1-6 Data Ingestion...
